# PredictShield AI - LSTM + Transformer Training (Sequence Models)

Trains BOTH sequence models on the **same training-ready dataset used by XGBoost and Isolation Forest**:

- **Bidirectional LSTM** - learns the ORDER of events over time
- **Transformer** - learns long-range dependencies via multi-head self-attention

**Data:** `cicids2017_training_ready.csv` - already StandardScaler-normalized (77 features) + integer labels (0-14). This is the SAME file XGBoost/IF used, so all models share identical features and label mapping (critical for the fusion layer).

**No IPs, no LabelEncoder, no StandardScaler needed** - the file is already preprocessed.

**Sequence:** sliding window of 50 consecutive flows (stride 10). The CSV is chronological, so row order = time order.

**Class weights** are used in the loss (15-class imbalance would otherwise collapse the model to BENIGN).

In [1]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}  GPUs: {torch.cuda.device_count()}')


PyTorch: 2.11.0+cu128
CUDA: True  GPUs: 2


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
print('imports OK')


imports OK


In [3]:
WINDOW = 50
STRIDE = 10
N_FEATURES = 77
N_CLASSES = 15
HIDDEN = 128
EPOCHS = 40
LR_LSTM = 0.001
LR_TRANS = 0.0003
WARMUP = 5
BATCH_SIZE = 64
RANDOM_STATE = 42
CLIP = 1.0

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


Device: cuda


In [4]:
# training_ready.csv = 77 scaled features + integer label (0-14), already preprocessed
CSV_PATH = '/kaggle/input/datasets/pritammandal143/training/cicids2017_training_ready.csv'

import os
print(f'File exists: {os.path.exists(CSV_PATH)}')
if os.path.exists(CSV_PATH):
    print(f'File size: {os.path.getsize(CSV_PATH)/1e6:.1f} MB')


File exists: True
File size: 4125.8 MB


In [5]:
# Load in float32 (features already scaled, label is int 0-14)
df = pd.read_csv(CSV_PATH, dtype=np.float32)
df['label'] = df['label'].astype(np.int32)
print(f'Loaded: {df.shape}')

label_col = 'label'
feature_cols = [c for c in df.columns if c != label_col]
print(f'Features: {len(feature_cols)}, Label: {label_col}')
print(f'Label range: {df[label_col].min()}..{df[label_col].max()}')


Loaded: (2827876, 78)
Features: 77, Label: label
Label range: 0..14


In [6]:
# Downsample BENIGN (label 0) to 200k, keep all attacks - same as XGBoost/IF
BENIGN_TARGET = 200_000
y_all = df[label_col].values
benign_idx = np.where(y_all == 0)[0]
attack_idx = np.where(y_all != 0)[0]

np.random.seed(RANDOM_STATE)
benign_sampled = np.random.choice(benign_idx, size=BENIGN_TARGET, replace=False)
keep_idx = np.sort(np.concatenate([benign_sampled, attack_idx]))

X = df[feature_cols].values[keep_idx].astype(np.float32)
y = y_all[keep_idx].astype(np.int64)
print(f'Downsampled: {X.shape[0]:,} rows (200k benign + {len(attack_idx):,} attacks)')
print(f'X shape: {X.shape}, y shape: {y.shape}')
del df


Downsampled: 756,556 rows (200k benign + 556,556 attacks)
X shape: (756556, 77), y shape: (756556,)


In [7]:
# Build sliding-window sequences: (N, WINDOW, 77), label = last flow's class
def build_sequences(X, y, window=WINDOW, stride=STRIDE):
    seqs, labels = [], []
    for start in range(0, len(X) - window + 1, stride):
        seqs.append(X[start:start + window])
        labels.append(y[start + window - 1])
    return np.stack(seqs), np.array(labels)

X_seq, y_seq = build_sequences(X, y)
print(f'Sequences: {X_seq.shape}, labels: {y_seq.shape}')
print(f'Label distribution:')
print(pd.Series(y_seq).value_counts().sort_index().to_string())


Sequences: (75651, 50, 77), labels: (75651,)
Label distribution:
0     20024
1       185
2     12802
3      1000
4     23015
5       550
6       590
7       797
8         3
9         4
10    15887
11      588
12      142
14       64


In [8]:
# Split 70/15/15 (NO stratify - rare classes have too few sequences,
# stratified split fails. Class weights already handle imbalance.)
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_seq, y_seq, test_size=0.30, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}')

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val, dtype=torch.long))
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


Train: 52,955  Val: 11,348  Test: 11,348


In [9]:
# Softened class weights: sqrt(inverse freq), normalized, clipped [0.5, 5.0]
# (v1 used raw inverse freq -> Heartbleed weight ~5000x, over-predicted rare classes)
import math
class_counts = pd.Series(y_train).value_counts()
w = {c: 1.0 / math.sqrt(class_counts.get(c, 1)) for c in range(N_CLASSES)}
mean_w = float(np.mean(list(w.values())))
w = {c: min(max(v / mean_w, 0.5), 5.0) for c, v in w.items()}
class_weights = torch.tensor([w[c] for c in range(N_CLASSES)], dtype=torch.float32, device=DEVICE)
print('Class weights (softened + clipped):')
for c in range(N_CLASSES):
    print(f'  class {c:2d}: {class_weights[c].item():.3f} (n={class_counts.get(c, 0)})')
criterion = nn.CrossEntropyLoss(weight=class_weights)


Class weights (softened + clipped):
  class  0: 0.500 (n=13963)
  class  1: 0.500 (n=137)
  class  2: 0.500 (n=8944)
  class  3: 0.500 (n=699)
  class  4: 0.500 (n=16134)
  class  5: 0.500 (n=404)
  class  6: 0.500 (n=412)
  class  7: 0.500 (n=564)
  class  8: 3.771 (n=2)
  class  9: 2.666 (n=4)
  class 10: 0.500 (n=11147)
  class 11: 0.500 (n=408)
  class 12: 0.550 (n=94)
  class 13: 5.000 (n=0)
  class 14: 0.813 (n=43)


## Model 1: Bidirectional LSTM

In [10]:
class BiLSTM(nn.Module):
    def __init__(self, in_features=N_FEATURES, hidden=HIDDEN, n_classes=N_CLASSES):
        super().__init__()
        self.lstm1 = nn.LSTM(in_features, hidden, batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(0.3)
        self.lstm2 = nn.LSTM(hidden * 2, hidden // 2, batch_first=True, bidirectional=True)
        self.dropout2 = nn.Dropout(0.3)
        self.fc1 = nn.Linear(hidden, 64)
        self.fc2 = nn.Linear(64, n_classes)
    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.dropout1(x)
        x, _ = self.lstm2(x)
        x = self.dropout2(x)
        x = x[:, -1, :]
        x = F.relu(self.fc1(x))
        return self.fc2(x)

lstm_model = BiLSTM().to(DEVICE)
optimizer_lstm = torch.optim.Adam(lstm_model.parameters(), lr=LR_LSTM)
print(lstm_model)


BiLSTM(
  (lstm1): LSTM(77, 128, batch_first=True, bidirectional=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (lstm2): LSTM(256, 64, batch_first=True, bidirectional=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=15, bias=True)
)


In [11]:
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out = model(xb)
            preds.append(out.argmax(dim=1).cpu())
            labels.append(yb.cpu())
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()

for epoch in range(EPOCHS):
    lstm_model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer_lstm.zero_grad()
        out = lstm_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), CLIP)
        optimizer_lstm.step()
        total_loss += loss.item()
    val_preds, val_labels = evaluate(lstm_model, val_loader)
    val_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS}  loss={total_loss/len(train_loader):.4f}  val_f1={val_f1:.4f}')


Epoch   1/40  loss=0.2133  val_f1=0.6441
Epoch   5/40  loss=0.0302  val_f1=0.8765
Epoch  10/40  loss=0.0164  val_f1=0.8952
Epoch  15/40  loss=0.0108  val_f1=0.9605
Epoch  20/40  loss=0.0068  val_f1=0.9429
Epoch  25/40  loss=0.0049  val_f1=0.9414
Epoch  30/40  loss=0.0044  val_f1=0.9575
Epoch  35/40  loss=0.0051  val_f1=0.9550
Epoch  40/40  loss=0.0035  val_f1=0.9571


In [12]:
test_preds, test_labels = evaluate(lstm_model, test_loader)
print('LSTM Results:')
print(f'  Accuracy : {accuracy_score(test_labels, test_preds):.4f}')
print(f'  F1 (macro): {f1_score(test_labels, test_preds, average="macro", zero_division=0):.4f}')
print(f'  F1 (weighted): {f1_score(test_labels, test_preds, average="weighted", zero_division=0):.4f}')
print()
print(classification_report(test_labels, test_preds, digits=3, zero_division=0))


LSTM Results:
  Accuracy : 0.9969
  F1 (macro): 0.9561
  F1 (weighted): 0.9969

              precision    recall  f1-score   support

           0      0.995     0.995     0.995      2983
           1      0.750     0.857     0.800        21
           2      1.000     1.000     1.000      1920
           3      0.982     1.000     0.991       164
           4      1.000     0.999     1.000      3486
           5      0.986     1.000     0.993        68
           6      1.000     0.990     0.995        99
           7      1.000     1.000     1.000       126
          10      1.000     0.997     0.998      2371
          11      0.938     0.962     0.950        79
          12      0.909     1.000     0.952        20
          14      0.889     0.727     0.800        11

    accuracy                          0.997     11348
   macro avg      0.954     0.961     0.956     11348
weighted avg      0.997     0.997     0.997     11348



## Model 2: Transformer

In [13]:
class TransformerModel(nn.Module):
    def __init__(self, in_features=N_FEATURES, d_model=128, n_heads=8,
                 n_layers=4, n_classes=N_CLASSES, dropout=0.2):
        super().__init__()
        self.embed = nn.Linear(in_features, d_model)
        self.pos_embed = nn.Embedding(WINDOW, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=512,
            dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.fc = nn.Linear(d_model, n_classes)
    def forward(self, x):
        x = self.embed(x)
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = x + self.pos_embed(positions)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.fc(x)

transformer_model = TransformerModel().to(DEVICE)
optimizer_t = torch.optim.Adam(transformer_model.parameters(), lr=LR_TRANS)
print(transformer_model)


TransformerModel(
  (embed): Linear(in_features=77, out_features=128, bias=True)
  (pos_embed): Embedding(50, 128)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=128, out_features=15, bias=True)
)


In [14]:
def lr_lambda(epoch):
    if epoch < WARMUP:
        return (epoch + 1) / WARMUP
    return 1.0

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer_t, lr_lambda)

for epoch in range(EPOCHS):
    transformer_model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer_t.zero_grad()
        out = transformer_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(transformer_model.parameters(), CLIP)
        optimizer_t.step()
        total_loss += loss.item()
    scheduler.step()
    val_preds, val_labels = evaluate(transformer_model, val_loader)
    val_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS}  loss={total_loss/len(train_loader):.4f}  val_f1={val_f1:.4f}  lr={optimizer_t.param_groups[0]["lr"]:.5f}')


Epoch   1/40  loss=0.3345  val_f1=0.4979  lr=0.00012
Epoch   5/40  loss=0.0495  val_f1=0.7796  lr=0.00030
Epoch  10/40  loss=0.0263  val_f1=0.8061  lr=0.00030
Epoch  15/40  loss=0.0178  val_f1=0.9365  lr=0.00030
Epoch  20/40  loss=0.0128  val_f1=0.9261  lr=0.00030
Epoch  25/40  loss=0.0114  val_f1=0.8498  lr=0.00030
Epoch  30/40  loss=0.0099  val_f1=0.9531  lr=0.00030
Epoch  35/40  loss=0.0093  val_f1=0.8513  lr=0.00030
Epoch  40/40  loss=0.0097  val_f1=0.9215  lr=0.00030


In [15]:
test_preds, test_labels = evaluate(transformer_model, test_loader)
print('Transformer Results:')
print(f'  Accuracy : {accuracy_score(test_labels, test_preds):.4f}')
print(f'  F1 (macro): {f1_score(test_labels, test_preds, average="macro", zero_division=0):.4f}')
print(f'  F1 (weighted): {f1_score(test_labels, test_preds, average="weighted", zero_division=0):.4f}')
print()
print(classification_report(test_labels, test_preds, digits=3, zero_division=0))


Transformer Results:
  Accuracy : 0.9937
  F1 (macro): 0.8357
  F1 (weighted): 0.9938

              precision    recall  f1-score   support

           0      0.991     0.990     0.990      2983
           1      0.739     0.810     0.773        21
           2      0.998     1.000     0.999      1920
           3      1.000     0.994     0.997       164
           4      0.999     0.999     0.999      3486
           5      1.000     1.000     1.000        68
           6      0.990     0.980     0.985        99
           7      0.962     1.000     0.981       126
           9      0.000     0.000     0.000         0
          10      0.999     0.995     0.997      2371
          11      0.855     0.899     0.877        79
          12      0.640     0.800     0.711        20
          14      0.714     0.455     0.556        11

    accuracy                          0.994     11348
   macro avg      0.838     0.840     0.836     11348
weighted avg      0.994     0.994     0.994    

In [16]:
torch.save(lstm_model.state_dict(), '/kaggle/working/lstm_cicids2017.pt')
torch.save(transformer_model.state_dict(), '/kaggle/working/transformer_cicids2017.pt')
print('Saved:')
print('  /kaggle/working/lstm_cicids2017.pt')
print('  /kaggle/working/transformer_cicids2017.pt')


Saved:
  /kaggle/working/lstm_cicids2017.pt
  /kaggle/working/transformer_cicids2017.pt


## Summary

- Both models trained on `cicids2017_training_ready.csv` (same file as XGBoost/IF)
- Already scaled + integer-labeled, so no LabelEncoder/StandardScaler needed
- Sliding window 50, stride 10
- Class weights for 15-class imbalance
- LSTM (BiLSTM 128->64) + Transformer (4 layers, 8 heads)
- Saved as `.pt` for fusion / `/ai/predict` integration


In [20]:

# ============ COMPLETE FIX: BiLSTM + TRANSFORMER -> ONNX ============

!pip install -q onnx onnxruntime

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort
import numpy as np

# Required variables must already exist:
# WINDOW, N_FEATURES, HIDDEN, N_CLASSES

assert WINDOW > 0 and N_FEATURES > 0
assert HIDDEN > 0 and N_CLASSES > 0

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)


# ---------------- 1. BiLSTM MODEL ----------------

class BiLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.lstm1 = nn.LSTM(
            N_FEATURES, HIDDEN,
            batch_first=True, bidirectional=True
        )
        self.dropout1 = nn.Dropout(0.3)

        self.lstm2 = nn.LSTM(
            HIDDEN * 2, HIDDEN // 2,
            batch_first=True, bidirectional=True
        )
        self.dropout2 = nn.Dropout(0.3)

        self.fc1 = nn.Linear(HIDDEN, 64)
        self.fc2 = nn.Linear(64, N_CLASSES)

    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.dropout1(x)

        x, _ = self.lstm2(x)
        x = self.dropout2(x)

        x = x[:, -1, :]
        x = F.relu(self.fc1(x))
        return self.fc2(x)


# ---------------- 2. TRANSFORMER MODEL ----------------

class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.embed = nn.Linear(N_FEATURES, 128)
        self.pos_embed = nn.Embedding(WINDOW, 128)

        layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            dim_feedforward=512,
            dropout=0.2,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=4,
            enable_nested_tensor=False
        )

        self.fc = nn.Linear(128, N_CLASSES)

    def forward(self, x):
        x = self.embed(x)

        positions = torch.arange(
            x.size(1), device=x.device
        ).unsqueeze(0)

        x = x + self.pos_embed(positions)
        x = self.encoder(x)
        x = x.mean(dim=1)

        return self.fc(x)


# ---------------- 3. EXPORT + VALIDATE + TEST ----------------

def export_and_test(model, checkpoint, output_file):
    checkpoint_path = os.path.join(OUT_DIR, checkpoint)
    output_path = os.path.join(OUT_DIR, output_file)

    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}"
        )

    print(f"\nLoading {checkpoint} ...")

    model = model.cpu()
    state = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )
    model.load_state_dict(state)
    model.eval()

    dummy = torch.randn(2, WINDOW, N_FEATURES)

    # Disable MHA fast-path to avoid
    # aten::_transformer_encoder_layer_fwd export error.
    old_fastpath = torch.backends.mha.get_fastpath_enabled()

    try:
        torch.backends.mha.set_fastpath_enabled(False)

        with torch.no_grad():
            torch.onnx.export(
                model,
                dummy,
                output_path,
                input_names=["input"],
                output_names=["logits"],
                dynamic_axes={
                    "input": {0: "batch"},
                    "logits": {0: "batch"}
                },
                opset_version=17,
                dynamo=False
            )
    finally:
        torch.backends.mha.set_fastpath_enabled(old_fastpath)

    # Validate ONNX graph
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model)

    # ONNX Runtime inference
    session = ort.InferenceSession(
        output_path,
        providers=["CPUExecutionProvider"]
    )

    onnx_output = session.run(
        ["logits"],
        {"input": dummy.numpy()}
    )[0]

    # Compare against PyTorch
    with torch.no_grad():
        pytorch_output = model(dummy).numpy()

    max_diff = float(
        np.max(np.abs(pytorch_output - onnx_output))
    )

    print(f"✅ Exported: {output_file}")
    print("ONNX validation: PASSED")
    print("PyTorch shape:", pytorch_output.shape)
    print("ONNX shape:", onnx_output.shape)
    print(f"Max absolute difference: {max_diff:.8f}")
    print("Predicted classes:", onnx_output.argmax(axis=1))
    print(f"File size: {os.path.getsize(output_path) / 1e6:.2f} MB")

    if not np.allclose(
        pytorch_output, onnx_output,
        rtol=1e-3, atol=1e-5
    ):
        print("⚠️ Warning: outputs differ beyond tolerance.")
    else:
        print("Numerical comparison: PASSED")


# ---------------- 4. EXPORT BOTH MODELS ----------------

export_and_test(
    BiLSTM(),
    "lstm_cicids2017.pt",
    "lstm_cicids2017.onnx"
)

export_and_test(
    TransformerModel(),
    "transformer_cicids2017.pt",
    "transformer_cicids2017.onnx"
)

print("\n🎉 DONE! Both ONNX models exported and tested.")
print("Saved in:", OUT_DIR)



Loading lstm_cicids2017.pt ...
✅ Exported: lstm_cicids2017.onnx
ONNX validation: PASSED
PyTorch shape: (2, 15)
ONNX shape: (2, 15)
Max absolute difference: 0.00000763
Predicted classes: [0 0]
File size: 1.55 MB
Numerical comparison: PASSED

Loading transformer_cicids2017.pt ...
✅ Exported: transformer_cicids2017.onnx
ONNX validation: PASSED
PyTorch shape: (2, 15)
ONNX shape: (2, 15)
Max absolute difference: 0.00000381
Predicted classes: [0 0]
File size: 3.35 MB
Numerical comparison: PASSED

🎉 DONE! Both ONNX models exported and tested.
Saved in: /kaggle/working
